In [1]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [2]:
import transformers
print(transformers.__version__)

5.3.0


In [3]:
!pip install -U accelerate

In [4]:
df = pd.read_csv("scaled.csv")

df = df[["title","Label"]]
df = df.dropna()

df.head()

,title,Label
0,Donald Trump endorsed an unabashedly noninterv...,1
1,While some Justice Department investigations a...,1
2,Trump Proudly Declares: Most Of The People I’v...,0
3,Donald Trump spent a day in January 2014 hobno...,1
4,Notable names include Ray Washburne (Commerce)...,1


In [5]:
df.head()

,title,Label
0,Donald Trump endorsed an unabashedly noninterv...,1
1,While some Justice Department investigations a...,1
2,Trump Proudly Declares: Most Of The People I’v...,0
3,Donald Trump spent a day in January 2014 hobno...,1
4,Notable names include Ray Washburne (Commerce)...,1


In [6]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["title"],
    df["Label"],
    test_size=0.2,
    random_state=42
)

In [7]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [8]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [9]:
class FakeNewsDataset(torch.utils.data.Dataset):
    
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

In [10]:
train_dataset = FakeNewsDataset(train_encodings, train_labels)
test_dataset = FakeNewsDataset(test_encodings, test_labels)

In [11]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch"
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [16]:
trainer.train()

c:\Users\alwin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.507428,0.403519
2,0.421798,0.424051
3,0.264777,0.318871


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\alwin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=41727, training_loss=0.4215142140379461, metrics={'train_runtime': 176379.0752, 'train_samples_per_second': 1.893, 'train_steps_per_second': 0.237, 'total_flos': 2.19572753474304e+16, 'train_loss': 0.4215142140379461, 'epoch': 3.0})

In [17]:
trainer.evaluate()

RuntimeError: on_train_begin must be called before on_evaluate

In [18]:
trainer.save_model("fake_news_model")
tokenizer.save_pretrained("fake_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fake_news_model\\tokenizer_config.json', 'fake_news_model\\tokenizer.json')

In [19]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("fake_news_model")
model = BertForSequenceClassification.from_pretrained("fake_news_model")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [20]:
import torch

text = "Government secretly controlling weather"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits)

print(prediction)

tensor(0)


In [21]:
label_map = {0: "Fake", 1: "True"}

print(label_map[prediction.item()])

Fake


In [22]:
def predict_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits).item()
    
    label_map = {0:"Fake", 1:"True"}
    
    return label_map[pred]

In [23]:
predict_news("Scientists discover new planet")

'Fake'

In [24]:
predict_news("Aliens landed in New York yesterday")
predict_news("NASA launches new satellite mission")
predict_news("Government secretly controlling weather")

'Fake'

In [25]:
predict_news("Aliens landed in New York yesterday")

'Fake'

In [26]:
predict_news("Government secretly controlling weather")

'Fake'

In [27]:
predict_news("NASA launches new satellite mission")

'True'